# CNN으로 MNIST 손글씨 분류하기

이 노트북은 **환경 확인 → 데이터 관찰 → CNN 정의 → 텐서 shape 확인 → 학습 → 평가 → 저장** 순서로 진행합니다. 위에서 아래로 셀을 하나씩 실행하세요.

> 처음에는 `QUICK_RUN = True`로 전체 흐름을 확인하고, 정상 실행된 뒤 `False`로 바꿔 더 많은 데이터로 학습하세요.

## 1. 라이브러리와 실행 설정

seed를 고정하고 CUDA가 보이면 GPU를 선택합니다. 모델과 입력 텐서는 반드시 같은 device에 있어야 합니다.

In [2]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
QUICK_RUN = True
EPOCHS = 1 if QUICK_RUN else 5
TRAIN_LIMIT = 2_000 if QUICK_RUN else 0  # 0은 전체 데이터
TEST_LIMIT = 1_000 if QUICK_RUN else 0
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
DATA_DIR = Path('/data')
OUTPUT_DIR = Path('/outputs/notebooks/cnn')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.4.1+cu121
device: cuda
GPU: NVIDIA GeForce RTX 3060


## 2. MNIST 데이터 준비

`ToTensor`는 이미지를 `(채널, 높이, 너비)` 텐서로 바꾸고, `Normalize`는 MNIST 평균과 표준편차를 이용해 픽셀 분포를 정돈합니다. 첫 실행 때만 `/data`에 데이터가 다운로드됩니다.

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

def take_subset(dataset, limit, seed):
    if limit <= 0 or limit >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)

train_dataset = take_subset(train_dataset, TRAIN_LIMIT, SEED)
test_dataset = take_subset(test_dataset, TEST_LIMIT, SEED + 1)

loader_options = dict(
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'),
)
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

images, labels = next(iter(train_loader))
print('train samples:', len(train_dataset))
print('test samples :', len(test_dataset))
print('image batch shape:', images.shape)
print('label batch shape:', labels.shape)

PermissionError: [Errno 13] Permission denied: '/data'

## 3. 모델에 넣기 전에 데이터 보기

정규화된 텐서를 다시 원래 밝기 범위에 가깝게 돌린 뒤 첫 10장을 확인합니다. 데이터와 라벨이 기대한 형태인지 먼저 보는 습관이 중요합니다.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for axis, image, label in zip(axes.flat, images[:10], labels[:10]):
    restored = image.squeeze(0) * 0.3081 + 0.1307
    axis.imshow(restored, cmap='gray', vmin=0, vmax=1)
    axis.set_title(f'label={label.item()}')
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. CNN 모델 정의

두 번의 합성곱과 pooling으로 이미지 특징을 추출하고, 펼친 특징을 두 개의 Linear 계층에 통과시켜 숫자 10개의 logit을 만듭니다.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(DEVICE)
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))

## 5. 계층을 지날 때 shape 추적

CNN 오류의 상당수는 Linear 계층에 들어가는 크기를 잘못 계산해서 생깁니다. 실제 배치 하나를 통과시키며 shape를 확인합니다.

In [ ]:
x = images[:4].to(DEVICE)
print('input'.ljust(22), tuple(x.shape))
for layer in model.features:
    x = layer(x)
    print(layer.__class__.__name__.ljust(22), tuple(x.shape))
for layer in model.classifier:
    x = layer(x)
    print(layer.__class__.__name__.ljust(22), tuple(x.shape))

## 6. 학습과 평가

한 batch마다 gradient 초기화 → 순전파 → 손실 → 역전파 → 가중치 갱신을 반복합니다. 평가할 때는 `eval()`과 `inference_mode()`로 가중치를 바꾸지 않습니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.inference_mode():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)
            logits = model(batch_images)
            total_loss += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_labels.size(0)

    train_loss = running_loss / len(train_dataset)
    test_loss, test_accuracy = evaluate(model, test_loader)
    history.append((train_loss, test_loss, test_accuracy))
    print(
        f'epoch={epoch:02d} train_loss={train_loss:.4f} '
        f'test_loss={test_loss:.4f} accuracy={test_accuracy:.2%}'
    )

## 7. 예측 결과 확인

정확도뿐 아니라 어떤 글씨를 틀리는지 직접 확인합니다. `true`와 `pred`가 다르면 오분류입니다.

In [ ]:
test_images, test_labels = next(iter(test_loader))
model.eval()
with torch.inference_mode():
    predictions = model(test_images.to(DEVICE)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(11, 4))
for axis, image, label, prediction in zip(
    axes.flat, test_images[:10], test_labels[:10], predictions[:10]
):
    restored = image.squeeze(0) * 0.3081 + 0.1307
    axis.imshow(restored, cmap='gray', vmin=0, vmax=1)
    color = 'green' if label.item() == prediction.item() else 'red'
    axis.set_title(f'true={label.item()}, pred={prediction.item()}', color=color)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 8. 모델 저장

가중치와 학습 설정을 `/outputs`에 저장합니다. 이 경로는 호스트의 `outputs/`와 연결되어 컨테이너를 종료해도 남습니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / 'mnist_cnn_notebook.pt'
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'seed': SEED,
    },
    'history': history,
}, checkpoint_path)
print('saved:', checkpoint_path)

## 다음 실험

1. `QUICK_RUN = False`로 바꾸고 정확도 변화를 확인하세요.
2. `EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE` 중 하나만 바꿔 결과를 비교하세요.
3. `Dropout(0.2)`를 제거했을 때 학습과 테스트 성능 차이가 어떻게 변하는지 확인하세요.
4. 첫 번째 합성곱의 출력 채널을 32에서 16으로 바꾸고 파라미터 수를 비교하세요.